    #   Pre-requisites

In [ ]:
import sys
# Define the path to the directory containing the module
module_dir = "../Main/RequiredFuntions"

# Append the directory to sys.path
sys.path.append(module_dir)

In [ ]:
import json
import time
import hashlib
import threading
import functions as fn
import dataTransfer as DT
import InitializationPhase as IP
import EncryptionDecryption as ED

    #   ACK-1

In [ ]:
with open('../Main/ReceivedData/keys.json', 'r') as file:
    keys = json.load(file)

with open('../Main/ReceivedData/primenumber.json', 'r') as file:
    primenumber = json.load(file)

In [ ]:
primenumber = primenumber["primenumber"]

In [ ]:
deviceUniqueId_j = "RajeshHomeDevice1"
deviceNonce_j = fn.nonce_gen()

In [ ]:
encryptedUniqueId = ED.symmetric_key_encryption(deviceUniqueId_j, 
                                                  "".encode(), 
                                                  keys["key"])

In [ ]:
compute_j = hashlib.sha256((deviceUniqueId_j + str(deviceNonce_j) + keys["key"]).encode()).hexdigest()

In [ ]:
DeviceData = {
    "encryptedUniqueId" : encryptedUniqueId,
    "deviceNonce" : deviceNonce_j,
    "compute" : compute_j
}

In [ ]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive, args=("device_received_data.json",))  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(DeviceData,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

In [ ]:
# step 2
print("Starting Step 2")
json_data = fn.read_json_file("../Main/ReceivedData/device_received_data.json")

In [ ]:
decryptedId = ED.symmetric_key_decryption(json_data["encryptedUniqueId"], 
                                                  keys["key"])

In [ ]:
decryptedId

In [ ]:
validationCompute_j = hashlib.sha256((decryptedId[0].decode() + 
                                      str(json_data["deviceNonce"]) +
                                      keys["key"]).encode()).hexdigest()

In [ ]:
validationCompute_j == json_data["compute"]

In [ ]:
gatewayUniqueId_j = "rajeshGateway"

In [ ]:
#generate nounce in gateway
gatewayNonce_j = fn.nonce_gen()

Secret_j = gatewayNonce_j ^ json_data["deviceNonce"]

In [ ]:
gatewayNonce_j

In [ ]:
Secret_j

In [ ]:
# shares generated by gateway
generatedRandomNumber = fn.nonce_gen()
sharesGeneratedByGateway = (Secret_j + (2 * int(generatedRandomNumber))) % primenumber

In [ ]:
secretIntegrity_j_partial = hashlib.sha256((str(Secret_j) + decryptedId[0].decode()).encode()).hexdigest()

In [ ]:
dict_storage = {
    gatewayUniqueId_j: {
            deviceUniqueId_j: {
               "partialSecretIntegrityUser": secretIntegrity_j_partial,
               "gatewayShare": sharesGeneratedByGateway
            }
        }
    }

dict_storage

In [ ]:
with open(f'./ReceivedData/datastore_device.json', 'w') as json_file:
        json.dump(dict_storage, json_file, indent=4)

In [ ]:
secretIntegrity_j = hashlib.sha256((secretIntegrity_j_partial + str(json_data["deviceNonce"])).encode()).hexdigest()

In [ ]:
# temporal identity to encrypt R and gatwayNonce

temporalIdentity_j = hashlib.sha256((decryptedId[0].decode() + keys["device_public"] + str(json_data["deviceNonce"]) ).encode()).hexdigest()

In [ ]:
encryptedRandomNumber = fn.xor_strings(str(generatedRandomNumber), temporalIdentity_j)
encryptedRandomNumber = fn.xor_strings(encryptedRandomNumber, keys["key"])

encryptedGatewayNonce = fn.xor_strings(str(gatewayNonce_j), temporalIdentity_j)
encryptedGatewayNonce = fn.xor_strings(encryptedGatewayNonce, keys["key"])

In [ ]:
gatewaySecondNonce_j = fn.nonce_gen()

temporalIdentityGateway = hashlib.sha256((gatewayUniqueId_j + keys["gateway_public"] + str(gatewaySecondNonce_j)).encode()).hexdigest()

compute_G = hashlib.sha256((
    temporalIdentityGateway + 
    secretIntegrity_j + 
    encryptedRandomNumber + 
    encryptedGatewayNonce + 
    str(gatewaySecondNonce_j)).encode()).hexdigest()

In [ ]:
deviceGatewayJson = {
    'secretIntegrity' : secretIntegrity_j,
    'encryptedRandomNumber' : encryptedRandomNumber,
    'encryptedGatewayNonce' : encryptedGatewayNonce,
    'gatwayNonce' : gatewaySecondNonce_j,
    'computeG' : compute_G
}

In [ ]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive, args=('received_data_device_gateway.json',))  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(deviceGatewayJson,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

In [ ]:
# step 3
print("Starting Step 3")
deviceGatewayJsonData = fn.read_json_file("../Main/ReceivedData/received_data_device_gateway.json")

In [ ]:
#compute temporal identity user
temporalIdentity_j_s3 = hashlib.sha256((deviceUniqueId_j + keys["device_public"] + str(deviceNonce_j)).encode()).hexdigest()

#compute temporal indentity gateway
temporalIdentity_j_g_s3 = hashlib.sha256((gatewayUniqueId_j + keys["gateway_public"] + str(deviceGatewayJsonData['gatwayNonce'])).encode()).hexdigest()

In [ ]:
validateComputeG = hashlib.sha256((temporalIdentity_j_g_s3 + 
                    deviceGatewayJsonData['secretIntegrity'] + 
                    str(deviceGatewayJsonData['encryptedRandomNumber']) + 
                    str(deviceGatewayJsonData['encryptedGatewayNonce']) + 
                    str(deviceGatewayJsonData['gatwayNonce'])).encode()).hexdigest()

In [ ]:
validateComputeG == deviceGatewayJsonData["computeG"]

In [ ]:
# unencrypt randomnumber and gateway nonce
GatewayRandomNumber = fn.xor_strings(deviceGatewayJsonData['encryptedRandomNumber'], temporalIdentity_j_s3)
GatewayRandomNumber = fn.xor_strings(GatewayRandomNumber, keys["key"])

In [ ]:
GatewayGeneratedNonce = fn.xor_strings(str(deviceGatewayJsonData['encryptedGatewayNonce']), temporalIdentity_j_s3)
GatewayGeneratedNonce = fn.xor_strings(GatewayGeneratedNonce, keys["key"])

In [ ]:
GatewayGeneratedNonce

In [ ]:
secretComputed = deviceNonce_j ^ int(GatewayGeneratedNonce)
deviceShare = (secretComputed + int(GatewayRandomNumber)) % primenumber

In [ ]:
validationSecretIntegrity = hashlib.sha256((str(secretComputed) + deviceUniqueId_j).encode()).hexdigest()
validationSecretIntegrity = hashlib.sha256( (validationSecretIntegrity + str(deviceNonce_j)).encode()).hexdigest()

In [ ]:
print(f"Secret Integrity validation : {validationSecretIntegrity == secretIntegrity_j}")

In [ ]:
with open('../Main/ReceivedData/datastore_device.json', 'r') as file:
    datastore = json.load(file)

In [ ]:
datastore[gatewayUniqueId_j][deviceUniqueId_j]["userShare"] = deviceShare
datastore[gatewayUniqueId_j][deviceUniqueId_j]["partialSecretIntegrityGateway"] = hashlib.sha256((str(secretComputed) + gatewayUniqueId_j).encode()).hexdigest()

In [ ]:
with open(f'./ReceivedData/datastore_device.json', 'w') as json_file:
        json.dump(datastore, json_file, indent=4)